### Frog

In [ ]:
import pandas as pd
import numpy as np

# =====================================================
# Input / output
# =====================================================

input_file = r"X:\Huihui\Frog\Analysis\03042026\Frog3\Brain1_1\virus_celltype_summary2.xlsx"
output_dir = r"X:\Huihui\Frog\Analysis\03042026\Frog3\Brain1_1"

# =====================================================
# Read Excel
# =====================================================

df = pd.read_excel(input_file)

# =====================================================
# Define genes used for classification
# =====================================================

genes_for_classification = ["Snap25", "slc1a2", "slc4a4"]

# Make sure these columns are numeric
df[genes_for_classification] = df[genes_for_classification].apply(
    pd.to_numeric, errors="coerce"
).fillna(0)

# =====================================================
# Threshold weak signals
# =====================================================

# Snap25 counts < 3 becomes 0
df.loc[df["Snap25"] < 3, "Snap25"] = 0

# slc1a2 and slc4a4 counts < 2 become 0
df.loc[df["slc1a2"] < 2, "slc1a2"] = 0
df.loc[df["slc4a4"] < 2, "slc4a4"] = 0

# =====================================================
# Calculate total counts of these three markers
# =====================================================

df["Snap25_slc1a2_slc4a4_total"] = (
    df["Snap25"] + df["slc1a2"] + df["slc4a4"]
)

# =====================================================
# Assign cell type
# =====================================================

def assign_cell_type(row):
    snap25 = row["Snap25"]
    slc1a2 = row["slc1a2"]
    slc4a4 = row["slc4a4"]

    total = row["Snap25_slc1a2_slc4a4_total"]

    # Rule 2: if total counts < 2, mark as NA
    if total < 2:
        return "NA"

    astro_max = max(slc1a2, slc4a4)

    # Find max count among Snap25, slc1a2, slc4a4
    marker_values = {
        "Snap25": snap25,
        "slc1a2": slc1a2,
        "slc4a4": slc4a4
    }

    max_count = max(marker_values.values())

    # If there is a tie for maximum, mark as unknown
    n_max = sum(v == max_count for v in marker_values.values())
    if n_max > 1:
        return "unknown"

    max_gene = max(marker_values, key=marker_values.get)

    # Rule 3: Snap25 is max
    if max_gene == "Snap25":
        ratio = astro_max / snap25

        if ratio < 0.67:
            return "Neuron"
        else:
            return "unknown"

    # Rule 4: slc1a2 or slc4a4 is max
    elif max_gene in ["slc1a2", "slc4a4"]:
        ratio = snap25 / astro_max

        if ratio < 0.67:
            return "astrocyte"
        else:
            return "unknown"

    else:
        return "unknown"


df["cell_type"] = df.apply(assign_cell_type, axis=1)

# =====================================================
# Count cell types by virus
# =====================================================

celltype_counts_by_virus = pd.crosstab(
    df["virus"],
    df["cell_type"]
)

# Make sure all expected cell types are included
expected_cell_types = ["NA", "Neuron", "astrocyte", "unknown"]

for cell_type in expected_cell_types:
    if cell_type not in celltype_counts_by_virus.columns:
        celltype_counts_by_virus[cell_type] = 0

celltype_counts_by_virus = celltype_counts_by_virus[expected_cell_types]


celltype_fraction_by_virus = celltype_counts_by_virus.div(
    celltype_counts_by_virus.sum(axis=1),
    axis=0
) * 100


print("Cell type counts by virus:")
print(celltype_counts_by_virus)

print("\nCell type percentage by virus:")
print(celltype_fraction_by_virus)

print("\nOverall cell type counts:")
print(df["cell_type"].value_counts())


df.to_excel(
    output_dir + r"\cells_with_neuron_astrocyte_assignment.xlsx",
    index=False
)

celltype_counts_by_virus.to_excel(
    output_dir + r"\neuron_astrocyte_counts_by_virus.xlsx"
)

celltype_fraction_by_virus.to_excel(
    output_dir + r"\neuron_astrocyte_fraction_by_virus_percent.xlsx"
)

Cell type counts by virus:
cell_type   NA  Neuron  astrocyte  unknown
virus                                     
EEEV       144     161         34       13
SFV         45      32         24       10
VEEV         1       0          1        0
WEEV         3       8          0        1
no_gene4    12       1          0        1

Cell type percentage by virus:
cell_type         NA     Neuron  astrocyte   unknown
virus                                               
EEEV       40.909091  45.738636   9.659091  3.693182
SFV        40.540541  28.828829  21.621622  9.009009
VEEV       50.000000   0.000000  50.000000  0.000000
WEEV       25.000000  66.666667   0.000000  8.333333
no_gene4   85.714286   7.142857   0.000000  7.142857

Overall cell type counts:
cell_type
NA           205
Neuron       202
astrocyte     59
unknown       25
Name: count, dtype: int64
